# Iteración 27: Implementación de Línea Base con Nixtla (MLForecast y StatsForecast)

## 🚀 ¿Qué es Nixtla y qué estamos buscando?
**Nixtla** es un ecosistema moderno y ultrarrápido diseñado específicamente para el modelado avanzado de series temporales. A diferencia de enfoques clásicos (donde debemos construir manualmente bucles complejos para calcular Lags, Medias Móviles y Ventanas Rodantes), Nixtla optimiza drásticamente el proceso de Feature Engineering temporal y permite acoplar modelos de Machine Learning (como CatBoost o LightGBM) en una única API eficiente.

En esta iteración buscamos **dos objetivos estratégicos principales**:
1. **Mantener el rigor de Datos de NB23:** Reutilizar toda la lógica avanzada de filtrado quirúrgico (exclusión de FLEET, 2020), densificación matemática (Dense Panel) e inyección de datos exógenos (Clima y Ciclismo) para no perder ni un ápice de calidad de datos.
2. **Agilidad Estructural y Modelos Intermitentes:** Dejar que `MLForecast` de Nixtla maneje la inercia temporal para la demanda Regular/Errática, y preparar el terreno para aplicar algoritmos puramente B2B (Croston, ADIDA, SBA) de `StatsForecast` a nuestras tribus Intermitentes.


### Paso 0: Preparación del Entorno S&OP con Nixtla
Cargamos las librerías base y el ecosistema Nixtla.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

from mlforecast import MLForecast
from mlforecast.target_transforms import Differences
from mlforecast.lag_transforms import RollingMean, RollingStd, ExponentiallyWeightedMean
from catboost import CatBoostRegressor
import lightgbm as lgb

from statsforecast import StatsForecast
from statsforecast.models import CrostonOptimized, CrostonSBA, ADIDA

sns.set_theme(style="whitegrid", palette="mako")
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

SEED = 42
DATA_DIR = Path('../Datasets/Datos Internos')
print('Librerías listas.')


Librerías listas.


### Paso 1: Inteligencia de Negocio y Funciones Base
Aplicamos el mismo calendario fijo y funciones de métricas validadas.


In [ ]:
ANIOS_TRAIN  = [2021, 2022, 2023]
ANIO_TEST    = 2024

FESTIVOS_FIJOS = [(1,1), (1,6), (5,1), (8,15), (10,12), (11,1), (12,6), (12,8), (12,25)]
VIERNES_SANTOS = {2020:'2020-04-10', 2021:'2021-04-02', 2022:'2022-04-15', 2023:'2023-04-07', 2024:'2024-03-29', 2025:'2025-04-18'}
MESES_ES = {'enero':1,'febrero':2,'marzo':3,'abril':4,'mayo':5,'junio':6,'julio':7,'agosto':8,'septiembre':9,'octubre':10,'noviembre':11,'diciembre':12}
REGION_MAP = {
    'GALICIA': 'Noroeste', 'ASTURIAS': 'Norte', 'CANTABRIA': 'Norte',
    'PAIS VASCO': 'Norte', 'NAVARRA': 'Norte', 'LA RIOJA': 'Norte',
    'ARAGON': 'Noreste', 'CATALUÑA': 'Noreste', 'ISLAS BALEARES': 'Noreste',
    'COMUNIDAD DE MADRID': 'Centro', 'CASTILLA Y LEON': 'Centro', 'CASTILLA-LA MANCHA': 'Centro', 'EXTREMADURA': 'Centro',
    'COMUNIDAD VALENCIANA': 'Este', 'REGION DE MURCIA': 'Sur', 'ANDALUCIA': 'Sur',
    'CANARIAS': 'Canarias', 'CEUTA': 'Sur', 'MELILLA': 'Sur',
}

def parse_fecha_es(s):
    try:
        _, resto = str(s).split(', ', 1)
        day, _, month_es, _, year = resto.strip().split()
        return pd.Timestamp(year=int(year), month=MESES_ES[month_es.lower()], day=int(day))
    except Exception:
        return pd.NaT

def global_wmape(y_true, y_pred):
    mask = y_true > 0
    if mask.sum() == 0: return np.nan
    return np.sum(np.abs(y_true[mask] - y_pred[mask])) / np.sum(y_true[mask]) * 100
print('Constantes parametrizadas.')


Constantes parametrizadas.


### Paso 2: Ingesta del 'Data Lake' Operativo
Absorbemos las transacciones rigurosamente y replicamos el purgado que demostró ser vital: eliminación de FLEET, aislamiento del año anómalo 2020 y vinculación de Clima y Ciclismo.


In [ ]:
df_raw = pd.read_excel(DATA_DIR / 'LineasAlbaranCliente.xlsx')
df_raw['fecha'] = df_raw['FechaAlbaran'].astype(str).apply(parse_fecha_es)
df_raw = df_raw.dropna(subset=['fecha'])
df_raw['anio']        = df_raw['fecha'].dt.isocalendar().year.astype(int)
df_raw['semana_anio'] = df_raw['fecha'].dt.isocalendar().week.astype(int)
df_raw['codigo_articulo'] = df_raw['CodigoArticulo'].astype(str).str.strip()
df_raw['Unidades']   = pd.to_numeric(df_raw['Unidades'], errors='coerce').fillna(0)
df_raw['ImporteNeto']= pd.to_numeric(df_raw['ImporteNeto'],errors='coerce').fillna(0)
df_raw['pct_desc2']  = pd.to_numeric(df_raw['%Descuento2'],errors='coerce').fillna(0)

df_art = pd.read_excel(DATA_DIR / 'MaestroArticulos.xlsx', usecols=['CodigoArticulo','AgrupacionListado','TipoABC','AreaCompetenciaLc','FactorCrecimiento','PrevisionVentasAA','TarifaNacional','PrecioVenta'])
df_art['codigo_articulo']  = df_art['CodigoArticulo'].astype(str).str.strip()
df_art['tipo_abc']         = df_art['TipoABC'].fillna('C').astype(str).str.upper().str[:1]

df_cli = pd.read_excel(DATA_DIR / 'MaestroClientes.xlsx', usecols=['CodigoCliente','Municipio','Provincia','CodigoNacion'])
df_can = pd.read_excel(DATA_DIR / 'Agrupacion Canales venta.xlsx', header=0)
df_can.columns = ['canal_raw','agrupacion_canal','tipo_agrupacion'] + list(df_can.columns[3:])

df_clima = pd.read_csv('../Datasets/clima_semanal_openmeteo.csv')
df_clima.columns = [c.lower() for c in df_clima.columns]
df_clima_nac = df_clima.groupby(['year','semana']).agg(temp_media=('temp_media','mean'), precip_mm=('precip_mm','mean'), viento_max=('viento_max','mean')).reset_index().rename(columns={'year':'anio','semana':'semana_anio'})

df_cicl = pd.read_excel('../Datasets/Calendario Ciclismo 22_24.xlsx')
df_cicl.columns = [c.strip() for c in df_cicl.columns]
df_cicl_agg = df_cicl.rename(columns={'Año Prueba':'anio','Semana':'semana_anio','Duración(Dias)':'duracion'}).groupby(['anio','semana_anio']).agg(num_pruebas_cicl=('anio','count'), dias_pruebas_cicl=('duracion','sum')).reset_index()
df_cicl_agg['hubo_prueba_cicl'] = 1
print('Data lake inicial ingerido.')


Data lake inicial ingerido.


### Paso 3: Filtrado Quirúrgico B2B y Agregación Semanal
Fusionamos canales, quitamos Export y cortamos operaciones anómalas (Covid y Grandes licitaciones FLEET).


In [ ]:
df_raw = df_raw.merge(df_can[['canal_raw','agrupacion_canal']], left_on='SerieAlbaran', right_on='canal_raw', how='left')
df_raw['agrupacion_canal'] = df_raw['agrupacion_canal'].fillna('Otros')
df_raw = df_raw.merge(df_cli[['CodigoCliente','CodigoNacion']], on='CodigoCliente', how='left')

df_es = df_raw[df_raw['CodigoNacion'] == 108].copy()
df_nac = df_es[df_es['agrupacion_canal'] != 'FLEET'].copy()
df_nac = df_nac[df_nac['anio'] >= 2021].copy()
print(f'Nacional sin FLEET (desde 2021): {len(df_nac):,} filas')

def get_festivos_espana(anios):
    festivos = set()
    for y in anios:
        for m, d in FESTIVOS_FIJOS:
            festivos.add((y, m, d))
        if y in VIERNES_SANTOS:
            vs = pd.Timestamp(VIERNES_SANTOS[y])
            festivos.add((vs.year, vs.month, vs.day))
    return festivos

def dias_laborables_iso(year, week, festivos_set):
    try: lunes = pd.Timestamp.fromisocalendar(int(year), int(week), 1)
    except ValueError: return 5
    return sum(1 for delta in range(5) if (lunes + pd.Timedelta(days=delta)).strftime('%Y-%m-%d') not in [f"{y}-{m:02d}-{d:02d}" for y,m,d in festivos_set])

festivos_set = get_festivos_espana(range(2021, 2026))
semanas_unicas = df_nac[['anio','semana_anio']].drop_duplicates().copy()
semanas_unicas['dias_laborables_semana'] = semanas_unicas.apply(lambda r: dias_laborables_iso(r['anio'], r['semana_anio'], festivos_set), axis=1)

GROUP_NAC = ['anio', 'semana_anio', 'codigo_articulo']
df_agg = df_nac.groupby(GROUP_NAC, as_index=False).agg(unidades=('Unidades','sum'))
df_agg['unidades'] = df_agg['unidades'].clip(lower=0)

def wmean_desc(g):
    w = g['Unidades'].abs()
    v = g['pct_desc2']
    denom = w.sum()
    return (v * w).sum() / denom if denom > 0 else 0.0

desc_agg = df_nac.groupby(GROUP_NAC).apply(wmean_desc).reset_index(name='por_descuento2')
df_agg = df_agg.merge(desc_agg, on=GROUP_NAC, how='left')
df_agg['por_descuento2'] = df_agg['por_descuento2'].fillna(0.0)
print('Transacciones agregadas a nivel SKU-Semana.')


Nacional sin FLEET (desde 2021): 453,846 filas


### Paso 4: Densificación de Panel e Inyección Exógena
Tal como realizamos en los Notebooks precursores, construimos una matriz ininterrumpida y enlazamos a ella la Meteorología y el Calendario Ciclista.


In [ ]:
semanas_grid = semanas_unicas[['anio', 'semana_anio']].copy()
semanas_grid['key'] = 1
skus_grid = pd.DataFrame({'codigo_articulo': df_nac['codigo_articulo'].unique(), 'key': 1})
dense_grid = semanas_grid.merge(skus_grid, on='key', how='outer').drop(columns=['key'])

df_agg = dense_grid.merge(df_agg, on=['codigo_articulo', 'anio', 'semana_anio'], how='left')
df_agg['unidades'] = df_agg['unidades'].fillna(0)
df_agg['por_descuento2'] = df_agg['por_descuento2'].fillna(0)

df_agg = df_agg.merge(semanas_unicas, on=['anio','semana_anio'], how='left')
df_agg = df_agg.merge(df_clima_nac, on=['anio','semana_anio'], how='left')
df_agg = df_agg.merge(df_cicl_agg, on=['anio','semana_anio'], how='left')

df_agg['num_pruebas_cicl']  = df_agg['num_pruebas_cicl'].fillna(0).astype(int)
df_agg['dias_pruebas_cicl'] = df_agg['dias_pruebas_cicl'].fillna(0)
df_agg['hubo_prueba_cicl']  = df_agg['hubo_prueba_cicl'].fillna(0).astype(int)

df_agg = df_agg.merge(df_art[['codigo_articulo','tipo_abc']], on='codigo_articulo', how='left')
df_agg['tipo_abc'] = df_agg['tipo_abc'].fillna('C')

print(f"Panel Denso Formado: {len(df_agg)} registros. Incorporando variables climáticas...")


### Paso 5: Formateo Integral a Nixtla (`unique_id`, `ds`, `y`)
Para disfrutar del Feature Engineering ultrarrápido de Nixtla, moldeamos el panel denso. Todas las columnas extraíbles (`temp_media`, `por_descuento2`) actuarán como *Dynamic Features* automáticas cuando llamemos a MLForecast.


In [ ]:
def date_from_iso(anio, semana):
    return pd.to_datetime(f'{anio}-W{semana:02d}-1', format='%G-W%V-%u')

df_agg['ds'] = df_agg.apply(lambda r: date_from_iso(r['anio'], r['semana_anio']), axis=1)

# Solución al Error de CatBoost: Codificamos 'tipo_abc' a numérico (A/B/C -> 0/1/2)
df_agg['tipo_abc_encoded'], abc_mapping = pd.factorize(df_agg['tipo_abc'])
print(f'Mapeo de Categorías tipo_abc: {dict(enumerate(abc_mapping))}')

df_nixtla = df_agg.rename(columns={'codigo_articulo': 'unique_id', 'unidades': 'y'})

# Usamos tipo_abc_encoded como feature para el modelo
df_nixtla = df_nixtla[['unique_id', 'ds', 'y', 'tipo_abc_encoded', 'dias_laborables_semana', 'por_descuento2', 
                       'temp_media', 'precip_mm', 'viento_max', 'num_pruebas_cicl', 'hubo_prueba_cicl']]
df_nixtla = df_nixtla.sort_values(['unique_id', 'ds']).reset_index(drop=True)
print(f"Dataset Final en Formato Nixtla preparado: {len(df_nixtla)} registros históricos.")


### Paso 6: Train/Test Cronológico
Separamos el año 2024 entero como bloque de evaluación (*Out of Sample*).


In [ ]:
cutoff = pd.to_datetime('2024-01-01')
train_df = df_nixtla[df_nixtla['ds'] < cutoff].copy()
test_df  = df_nixtla[df_nixtla['ds'] >= cutoff].copy()

print(f"Train (Historico): {len(train_df)} rows. Test (2024 Ciego): {len(test_df)} rows.")


### Paso 7: Modelado con MLForecast (Características Dinámicas e Inerciales Automáticas)
Pasamos al regresor `CatBoost` el objeto `MLForecast`. Fíjate que configuramos `lags` y `lag_transforms`. Nixtla creará todas estas columnas en tiempo de entrenamiento y, durante el `predict()`, será capaz de usarlas de forma autoregresiva (predice la semana 1, calcula su lag para predecir la semana 2...).


In [ ]:
models = [
    CatBoostRegressor(iterations=300, learning_rate=0.08, depth=5, silent=True, random_seed=SEED)
]

fcst = MLForecast(
    models=models,
    freq='W-MON',
    lags=[1, 4, 12, 52], # Lags: Semana anterior, Mes anterior, Trimestre anterior, Año Pasado
    lag_transforms={
        1: [RollingMean(window_size=4), RollingStd(window_size=4)],
        4: [RollingMean(window_size=12)],
    },
    date_features=['month', 'week']
)

print("Entrenando CatBoost con Nixtla (aprox 15-30s). MLForecast procesará variables exógenas como features dinámicas automáticas...")
# Las variables estáticas categóricas codificadas se declaran como static_features:
fcst.fit(train_df, static_features=['tipo_abc_encoded'])
print("Entrenamiento listo.")


### Paso 8: Forecasting y Representación Visual Táctica
Atención: Para predecir el futuro con variables que fluctúan temporalmente (Ej: `temp_media` o `precip_mm` del 2024), **StatsForecast/MLForecast** exigen que pases un dataframe auxiliar (`X_df`) con las fechas futuras y el valor de esas variables exógenas.
Afortunadamente, nuestro `test_df` ya cubre esto perfectamente.


In [ ]:
# Features Dinámicas que el modelo necesita saber del futuro (clima, eventos, dtos previstos)
future_exog = test_df.drop(columns=['y', 'tipo_abc_encoded']) 
horizonte_semanas = test_df['ds'].nunique()

preds = fcst.predict(horizon=int(horizonte_semanas), X_df=future_exog)
test_resultados = test_df[['unique_id','ds','y','tipo_abc_encoded']].merge(preds, on=['unique_id','ds'], how='inner')
test_resultados['CatBoostRegressor'] = test_resultados['CatBoostRegressor'].clip(lower=0)

wmape_val = global_wmape(test_resultados['y'], test_resultados['CatBoostRegressor'])
print(f"\n★ WMAPE Global (Nixtla + CatBoost + Densificado + Clima) para 2024: {wmape_val:.2f}%")

# Visual S&OP
test_macro = test_resultados.groupby('ds').agg({'y':'sum', 'CatBoostRegressor':'sum'}).reset_index()
plt.figure(figsize=(14, 6))
sns.lineplot(data=test_macro, x='ds', y='y', label='Real B2B (2024)', linewidth=2.5, color='orange')
sns.lineplot(data=test_macro, x='ds', y='CatBoostRegressor', label='Predicción Nixtla Baseline', linewidth=2, linestyle='--', color='teal')
plt.title('Validación de Serie Temporal Consolidada CRUZBER (Nixtla)', fontsize=16, fontweight='bold')
plt.ylabel('Unidades Vendidas (Nacional)')
plt.xlabel('Semanas ISO')
plt.legend()
plt.tight_layout()
plt.show()


### Paso 9: Rumbo a la Siguiente Iteración
Hemos logrado igualar el pipeline ultra-complejo de nuestro Baseline original (Iteración 23) a través de los atajos y limpiezas optimizadas del framework `Nixtla`.

**¿Cuál es la siguiente fase?**
Utilizar `StatsForecast` aplicando clasificaciones *Intermitentes* (`CrostonSBA`, `ADIDA`) por id único para sustituir el Catboost genérico en aquellos repuestos o modelos donde la venta media se estanca a 0 recurrentemente.
